In [ ]:
# =============================================================================
# HOW DO WE GRADE A FAKE PHOTO? — IS, FID, CLIP Score
# =============================================================================
#
# GAN / VAE / diffusion can all spit out pictures. The human test is easy:
#   "Does this look like a real cat?"
# Machines need a NUMBER, because we cannot eyeball 50,000 fakes.
#
# Pixel MSE is a bad grade for art. Two different photos of "a red cat"
# can have huge pixel error and still both be good. So we grade in the
# BRAIN of a frozen classifier (Inception or CLIP), not in raw pixels.
#
#
# ---------------------------------------------------------------------------
# 1) INCEPTION SCORE (IS) — "is it a clear SOMETHING, and many kinds?"
# ---------------------------------------------------------------------------
#
# Pretend Inception-v3 is a museum guard who names ImageNet classes
# (1000 labels: tabby, sports car, …). You show it a pile of fakes.
#
# For ONE image it outputs p(y | image) — 1000 probabilities.
#
# Two wishes, in English:
#
#   (A) SHARP / CONFIDENT
#       One fake should look like ONE class, not "maybe cat, maybe toaster."
#       Guard says: p(y | image) is PEEKED (one big number, rest tiny).
#
#   (B) DIVERSE
#       The whole pile should cover MANY classes, not 10,000 identical cats.
#       Guard's average guess p(y) should be spread out, not one spike.
#
# IS combines them with KL divergence, then exp():
#
#   IS = exp(  average over images of  KL( p(y|image)  ||  p(y) )  )
#
#   High IS  → each image is a confident class AND the set is varied.
#   Low IS   → mush, or 10,000 copies of the same face.
#
# Rough scale (ImageNet-trained Inception, big sets):
#   random noise ~ 1     CIFAR-ish GANs ~ 2–8     strong ImageNet models >> 10
#
# What IS cannot see:
#   • It never looks at REAL photos. A generator that only makes "perfect
#     ImageNet posters" can score high even if it missed your dataset.
#   • It only knows 1000 ImageNet labels. "Van Gogh fruit bowl" is not a class.
#   • Mode collapse can still sneak through if those few modes look confident.
#
#
# ---------------------------------------------------------------------------
# 2) FRÉCHET INCEPTION DISTANCE (FID) — "do fakes live in the same
#    neighborhood as REAL photos?"
# ---------------------------------------------------------------------------
#
# This is the usual paper number. Lower is better (a DISTANCE).
#
# Picture two clouds of points:
#   Real photos  → run through Inception, grab a mid-layer vector
#                  (not the 1000 labels — a 2048-D "smell" of the image)
#   Fake photos  → same
#
# Fit a blob (Gaussian) to each cloud: mean μ and covariance Σ.
# FID = how far those two blobs are (Fréchet / Wasserstein-2 between Gaussians):
#
#   FID = ||μ_real − μ_fake||²  +  trace( Σ_real + Σ_fake − 2√(Σ_real Σ_fake) )
#
# English:
#   first term  = "are the averages in the same place?"
#   second term = "are the spreads the same shape?"
#
#   FID ≈ 0     fakes indistinguishable from reals (in this feature space)
#   FID small   good (CIFAR papers brag about low tens or less)
#   FID huge    fakes in a different neighborhood (blur, wrong colors, collapse)
#
# Why better than IS for "does it match my data?":
#   FID USES the real set. IS does not.
#
# Gotchas:
#   • Need LOTS of images (papers use 10k–50k). A handful of fakes = noisy FID.
#   • Same Inception preprocess (299×299, ImageNet mean/std) or numbers lie.
#   • Not a human. Two clouds can match while pictures still look weird.
#
#
# ---------------------------------------------------------------------------
# 3) CLIP SCORE — "does the picture MATCH THE PROMPT?"
# ---------------------------------------------------------------------------
#
# IS/FID never read your text. A beautiful mountain when you asked for
# "a cat" still can look "real." CLIP Score grades TEXT–IMAGE agreement.
#
# CLIP (from the Stable Diffusion notebook) has two frozen encoders:
#   image → vector     text → vector     (same vector space)
#
# CLIP Score ≈ cosine similarity (or a scaled version) between:
#   CLIP(image)  and  CLIP("a watercolor cat on a windowsill")
#
#   High  → picture and words point the same way
#   Low   → pretty image, wrong homework
#
# Use it for text-to-image (Stable Diffusion). IS/FID still useful for
# "looks like ImageNet / CIFAR," not for "obeyed the prompt."
#
#
# ---------------------------------------------------------------------------
# One table
# ---------------------------------------------------------------------------
#   Metric     Asks                         Needs reals?   Higher/lower better?
#   -------    ---------------------------  -------------   ---------------------
#   IS         clear + diverse classes      no             HIGHER
#   FID        fake cloud ≈ real cloud      YES            LOWER
#   CLIP       image matches the prompt     no (needs text) HIGHER
#
# Next cells: load Inception, compute IS, then FID, then CLIP on toy images.
#


In [2]:
# Setup

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torchvision.models import inception_v3, Inception_V3_Weights
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seed
torch.manual_seed(42)

/Users/girish11/aifromscratch_code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0827 01:18:18.077000 84660 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0827 01:18:18.111000 84660 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0827 01:18:18.135000 84660 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Using device: cpu


In [3]:
# Inception Score Implementation
def load_inception():
    model = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False)
    model.eval()
    model.to(device)
    return model

inception = load_inception()

# The transform_input=False allows us to use our own preprocessing. We need to resize images to 299×299 and normalize according to ImageNet statistics.

inception_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /Users/girish11/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:04<00:00, 25.2MB/s] 


In [ ]:
# Now implement the Inception Score. We'll compute probabilities in batches, then calculate the score.
def inception_score(images, batch_size=32, splits=10):
    """
    images: list of PIL Images or a tensor of shape [N, 3, H, W] in [0,1] range.
    Returns mean and std of IS over splits.
    """
    if isinstance(images, torch.Tensor):
        # Assume images are already preprocessed and on device
        pass
    else:
        # Convert list of PIL to tensor
        tensors = []
        for img in images:
            img_tensor = inception_transform(img).unsqueeze(0)
            tensors.append(img_tensor)
        images = torch.cat(tensors, dim=0)
    images = images.to(device)

    N = images.size(0)
    preds = []
    with torch.no_grad():
        for i in range(0, N, batch_size):
            batch = images[i:i+batch_size]
            logits = inception(batch)
            probs = F.softmax(logits, dim=1)
            preds.append(probs.cpu())
    preds = torch.cat(preds, dim=0)

    # Split into splits
    split_scores = []
    for k in range(splits):
        part = preds[k * (N // splits): (k+1) * (N // splits)]
        py = part.mean(dim=0)  # marginal p(y)
        kl = part * (torch.log(part + 1e-10) - torch.log(py + 1e-10))
        kl_div = kl.sum(dim=1).mean()
        split_scores.append(torch.exp(kl_div).item())
    return np.mean(split_scores), np.std(split_scores)